DINO feature space visualization (UMAP / t-SNE)

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from sklearn.manifold import TSNE
import umap
%matplotlib inline

REPO = Path("..").resolve()
sys.path.insert(0, str(REPO))

In [ ]:
FEATURES_DIR     = REPO / "outputs/notebook/encode_coco_gt_normalized"
OUT_DIR          = REPO / "outputs/notebook/viz/dino_normalized"

# OWOD: plot only newly introduced classes for this task (1–4). None = all GT classes
SPLIT_TASK       = 2
SPLIT_FILE       = REPO / "data/OWDETR/VOC2007/ImageSets/t2_train.txt"  

LABEL_SOURCE     = "gt"       # "gt" | "none"
METHODS          = ["umap", "tsne"]
PLOT_DIM         = 2
SEED             = 42
MAX_POINTS       = 3000
MAX_LEGEND       = 20
USE_STRATIFIED_SAMPLE = True
PER_CLASS_MAX    = 150

# UMAP
UMAP_NEIGHBORS   = 30
UMAP_MIN_DIST    = 0.25
UMAP_SPREAD      = 1.5
UMAP_METRIC      = "cosine"
UMAP_N_EPOCHS    = 800
USE_SUPERVISED_UMAP = False

# t-SNE
TSNE_PERPLEXITY  = 45.0
TSNE_ITERS       = 2500
TSNE_EARLY_EXAGGERATION = 12.0
TSNE_LEARNING_RATE = 200.0

# Plot
FIGSIZE          = (12, 10)
SCATTER_SIZE     = 7
SCATTER_ALPHA    = 0.5
SAVE_EXT         = "pdf"    

MIN_POINTS = {"umap": 3, "tsne": 4}
METHOD_TITLE = {"umap": "UMAP", "tsne": "t-SNE"}

In [ ]:
def set_axes3d_equal_aspect(ax, xyz: np.ndarray) -> None:
    pts = np.asarray(xyz, dtype=float)
    if pts.ndim != 2 or pts.shape[0] == 0 or pts.shape[1] < 3:
        return
    pts = pts[:, :3]
    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    span = float((maxs - mins).max())
    if span <= 0:
        span = 1.0
    ctr = (mins + maxs) * 0.5
    half = span * 0.5
    ax.set_xlim(ctr[0] - half, ctr[0] + half)
    ax.set_ylim(ctr[1] - half, ctr[1] + half)
    ax.set_zlim(ctr[2] - half, ctr[2] + half)
    try:
        ax.set_box_aspect((1, 1, 1))
    except AttributeError:
        pass


def set_axes2d_equal_centered(ax, xy: np.ndarray) -> None:
    pts = np.asarray(xy, dtype=float)
    if pts.ndim != 2 or pts.shape[0] == 0 or pts.shape[1] < 2:
        return
    pts = pts[:, :2]
    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    span = float((maxs - mins).max())
    if span <= 0:
        span = 1.0
    ctr = (mins + maxs) * 0.5
    half = span * 0.5
    ax.set_xlim(ctr[0] - half, ctr[0] + half)
    ax.set_ylim(ctr[1] - half, ctr[1] + half)
    ax.set_aspect("equal", adjustable="box")


def sample_indices(n: int, k: int, seed: int) -> np.ndarray:
    if k <= 0 or k >= n:
        return np.arange(n, dtype=np.int64)
    idx = np.random.default_rng(seed).choice(n, k, replace=False)
    idx.sort()
    return idx.astype(np.int64, copy=False)


def stratified_sample_indices(labels: list, k: int, seed: int, per_class_max: int = 0) -> np.ndarray:
    # Sample up to k points
    n = len(labels)
    if k <= 0 or k >= n:
        return np.arange(n, dtype=np.int64)
    by_cls: dict[str, list[int]] = {}
    for j, lab in enumerate(labels):
        by_cls.setdefault(lab, []).append(j)
    n_cls = max(1, len(by_cls))
    per = max(1, k // n_cls)
    if per_class_max > 0:
        per = min(per, per_class_max)
    rng = np.random.default_rng(seed)
    picked: list[int] = []
    for js in by_cls.values():
        js_arr = np.array(js, dtype=np.int64)
        take = min(len(js_arr), per)
        picked.extend(rng.choice(js_arr, take, replace=False).tolist())
    if len(picked) > k:
        picked = rng.choice(np.array(picked, dtype=np.int64), k, replace=False).tolist()
    elif len(picked) < k:
        rest = np.setdiff1d(np.arange(n, dtype=np.int64), np.array(picked, dtype=np.int64))
        extra = min(k - len(picked), len(rest))
        if extra > 0:
            picked.extend(rng.choice(rest, extra, replace=False).tolist())
    out = np.array(sorted(picked), dtype=np.int64)
    return out


def reduce_embedding(
    method: str,
    x: np.ndarray,
    plot_dim: int,
    seed: int,
    *,
    y_cls: np.ndarray | None = None,
    umap_neighbors: int = 30,
    umap_min_dist: float = 0.25,
    umap_spread: float = 1.5,
    umap_metric: str = "cosine",
    umap_n_epochs: int = 800,
    use_supervised_umap: bool = False,
    tsne_perplexity: float = 45.0,
    tsne_iters: int = 2500,
    tsne_early_exaggeration: float = 12.0,
    tsne_learning_rate: float | str = 200.0,
) -> np.ndarray:
    if method == "umap":
        n_neighbors = min(umap_neighbors, max(2, x.shape[0] - 1))
        reducer = umap.UMAP(
            n_components=plot_dim,
            n_neighbors=n_neighbors,
            min_dist=umap_min_dist,
            spread=umap_spread,
            metric=umap_metric,
            n_epochs=umap_n_epochs,
            random_state=seed,
        )
        if use_supervised_umap and y_cls is not None:
            return reducer.fit_transform(x, y_cls)
        return reducer.fit_transform(x)

    if method == "tsne":
        if x.shape[0] < 4:
            raise ValueError("t-SNE requires at least 4 points.")
        perplexity = min(tsne_perplexity, max(5.0, float(x.shape[0] - 1) / 3.0))
        lr = tsne_learning_rate
        if isinstance(lr, (int, float)) and lr <= 0:
            lr = "auto"
        reducer = TSNE(
            n_components=plot_dim,
            perplexity=perplexity,
            max_iter=tsne_iters,
            early_exaggeration=tsne_early_exaggeration,
            learning_rate=lr,
            random_state=seed,
            init="pca",
            metric="euclidean",
        )
        return reducer.fit_transform(x)

    raise ValueError(f"Unknown reduction method: {method}")

In [ ]:
def prepare_labels(records, source):
    if source == "none":
        return ["UNLABELED"] * len(records)
    if source == "gt":
        return [str(r.get("gt_category") or "UNLABELED") for r in records]
    raise ValueError(f"Unsupported LABEL_SOURCE: {source!r}")


def l2_normalize_rows(x, eps=1e-12):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, eps)


def reduce(method, x, y_cls=None):
    return reduce_embedding(
        method, x, PLOT_DIM, SEED,
        y_cls=y_cls,
        umap_neighbors=UMAP_NEIGHBORS,
        umap_min_dist=UMAP_MIN_DIST,
        umap_spread=UMAP_SPREAD,
        umap_metric=UMAP_METRIC,
        umap_n_epochs=UMAP_N_EPOCHS,
        use_supervised_umap=USE_SUPERVISED_UMAP,
        tsne_perplexity=TSNE_PERPLEXITY,
        tsne_iters=TSNE_ITERS,
        tsne_early_exaggeration=TSNE_EARLY_EXAGGERATION,
        tsne_learning_rate=TSNE_LEARNING_RATE,
    )


def save_fig(fig, path, dpi=220):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=dpi)
    plt.show()


def plot_features_only(points, color, title, out_path, *, categorical,
                       label_names=None, counts=None, max_legend=20):
    fig = plt.figure(figsize=FIGSIZE)
    ax = fig.add_subplot(111, projection="3d" if PLOT_DIM == 3 else None)
    cmap = "tab20" if categorical else "viridis"
    if PLOT_DIM == 3:
        sc = ax.scatter(
            points[:, 0], points[:, 1], points[:, 2],
            c=color, cmap=cmap, s=SCATTER_SIZE, alpha=SCATTER_ALPHA,
        )
        set_axes3d_equal_aspect(ax, points[:, :3])
    else:
        sc = ax.scatter(
            points[:, 0], points[:, 1],
            c=color, cmap=cmap, s=SCATTER_SIZE, alpha=SCATTER_ALPHA,
        )
        set_axes2d_equal_centered(ax, points[:, :2])
    if color is not None:
        if categorical and label_names and counts:
            top = [lab for lab, _ in counts.most_common(max_legend)]
            handles = [
                Line2D([0], [0], marker="o", color="w",
                       markerfacecolor=sc.cmap(sc.norm(label_names.index(lab))),
                       markersize=6, label=f"{lab} ({counts[lab]})") for lab in top
            ]
            ax.legend(handles=handles, loc="upper right", fontsize=8, title="Classes")
        else:
            fig.colorbar(sc, ax=ax)
    ax.set_title(title)
    save_fig(fig, out_path, dpi=200)


def resolve_path(path: str | Path) -> Path:
    path = Path(path)
    return path if path.is_absolute() else (REPO / path).resolve()

In [ ]:
from scripts.utils import filter_records_by_split
from src.evaluation.owod_split import COCO_CLASSES, TASK_SPLITS

feat_dir = resolve_path(FEATURES_DIR)
features = np.load(feat_dir / "features.npy").astype(np.float32)
records = json.loads((feat_dir / "records.json").read_text())
records.sort(key=lambda r: int(r["feature_index"]))

if SPLIT_FILE:
    records = filter_records_by_split(records, str(SPLIT_FILE), log_fn=print)

if SPLIT_TASK is not None:
    new_class_names = {COCO_CLASSES[i] for i in TASK_SPLITS[int(SPLIT_TASK)]}
    records = [r for r in records if r.get("gt_category") in new_class_names]
    print(
        f"OWOD task {SPLIT_TASK} new classes: {len(records)} records, "
        f"{len(new_class_names)} classes"
    )
    print("classes:", ", ".join(sorted(new_class_names)))

out_dir = Path(OUT_DIR).expanduser().resolve()
out_dir.mkdir(parents=True, exist_ok=True)

feature_labels = prepare_labels(records, LABEL_SOURCE)
valid = [(i, int(r["feature_index"])) for i, r in enumerate(records)
         if 0 <= int(r["feature_index"]) < features.shape[0]]
feat_idx = np.array([fi for _, fi in valid], dtype=np.int64)
feat = l2_normalize_rows(features[feat_idx].astype(np.float32, copy=False))
lab = [feature_labels[i] for i, _ in valid]
scores = np.array([records[i].get("score", 0.0) for i, _ in valid], dtype=np.float32)

if USE_STRATIFIED_SAMPLE:
    si = stratified_sample_indices(lab, MAX_POINTS, SEED, per_class_max=PER_CLASS_MAX)
else:
    si = sample_indices(feat.shape[0], MAX_POINTS, SEED)
feat = feat[si]
lab = [lab[i] for i in si]
scores = scores[si]
class_sorted = sorted(set(lab))
y_cls = np.array([class_sorted.index(x) for x in lab], dtype=np.int32)
print(f"plotted sample: {feat.shape[0]} points, {len(class_sorted)} classes")
print(Counter(lab).most_common(5), "...")

In [ ]:
# Plot UMAP / t-SNE
use_class = len(set(lab)) > 1
if use_class:
    class_sorted = sorted(set(lab))
    color = np.array([class_sorted.index(x) for x in lab], dtype=np.int32)
    counts = Counter(lab)
    split_tag = f"t{SPLIT_TASK}-new, " if SPLIT_TASK is not None else ""
    label_names = class_sorted
else:
    class_sorted, counts, label_names = None, None, None
    color = scores
    split_tag = ""

for m in METHODS:
    if use_class:
        sup_tag = "supervised UMAP, " if (m == "umap" and USE_SUPERVISED_UMAP) else ""
        color_title = f"{split_tag}{sup_tag}n={len(class_sorted)} classes"
    else:
        color_title = "score-colored"
    if len(feat) < MIN_POINTS[m]:
        print(f"skip {m}: n={len(feat)} < {MIN_POINTS[m]}")
        continue
    pts = reduce(m, feat, y_cls=y_cls)
    plot_features_only(
        pts, color, f"{METHOD_TITLE[m]} {PLOT_DIM}D ({color_title})",
        out_dir / f"{m}_{PLOT_DIM}d.{SAVE_EXT}",
        categorical=use_class, label_names=label_names, counts=counts,
        max_legend=MAX_LEGEND,
    )
    print(f"saved {m}_{PLOT_DIM}d.{SAVE_EXT}")